# Advanced Python Pickling — Problems with Complete Solutions

This notebook develops advanced practical skill with Python's `pickle` module.

## Learning goals

By the end, you should be able to:

- serialize and deserialize non-trivial object graphs;
- reason about equality, identity, aliases, and cycles;
- understand the pickle memo and protocol choices;
- customize object state with `__getstate__` / `__setstate__`;
- migrate older serialized state when classes evolve;
- inspect pickle bytecode safely with `pickletools`;
- build a restricted unpickler for narrow trusted use cases;
- authenticate pickle bytes before loading them;
- write pickle files atomically;
- stream multiple pickled objects;
- externalize selected objects with persistent IDs;
- combine pickle with compression;
- benchmark serialization choices;
- design a robust local checkpoint format.

> **Security rule:** Never unpickle data from an untrusted or unauthenticated source.  
> Pickle is a Python object reconstruction format, not a safe interchange format.


## Setup

The examples intentionally use only the Python standard library so the notebook is portable.


In [ ]:
from __future__ import annotations

import builtins
import bz2
import gzip
import hashlib
import hmac
import io
import os
import pickle
import pickletools
import statistics
import tempfile
import timeit
from dataclasses import dataclass, field
from pathlib import Path
from pprint import pprint
from typing import Any

print("Python pickle highest protocol:", pickle.HIGHEST_PROTOCOL)


## Security first: a safe demonstration

A pickle can instruct Python to call global objects while reconstructing data. That is why **loading** an untrusted pickle can execute code.

The next cells inspect a deliberately suspicious pickle **without loading it**. The payload only references `builtins.print`, so even if someone accidentally loads it, it merely prints a harmless message.


In [ ]:
class HarmlessDemo:
    def __reduce__(self):
        return (builtins.print, ("[demo] pickle reconstruction can call functions",))

demo_blob = pickle.dumps(HarmlessDemo(), protocol=pickle.HIGHEST_PROTOCOL)

print("Serialized bytes:", len(demo_blob))
pickletools.dis(demo_blob)


### Best practice

Use `pickletools.dis(...)` to inspect questionable pickle bytecode **without calling `pickle.loads`**. Inspection is not a general malware detector, but it is safer than blindly deserializing.

For data crossing trust boundaries, prefer formats such as JSON, MessagePack, Protocol Buffers, or another format whose parser does not reconstruct arbitrary Python objects.


# Problem 1 — Equality vs identity after a round trip

Create a nested object, pickle it, then unpickle it.

Determine which of these should be true:

1. `original == restored`
2. `original is restored`
3. `original["numbers"] is restored["numbers"]`

Then verify your predictions.


In [ ]:
# Starter data
original = {
    "name": "advanced-pickle",
    "numbers": [1, 2, 3, 5, 8],
    "meta": {"active": True, "version": 1},
}


## Solution 1

Pickle reconstructs a new object graph. Equivalent values usually compare equal, but the newly loaded top-level and nested mutable objects are different objects from the originals.


In [ ]:
blob = pickle.dumps(original, protocol=pickle.HIGHEST_PROTOCOL)
restored = pickle.loads(blob)

checks = {
    "equal_value": original == restored,
    "same_top_level_identity": original is restored,
    "same_nested_list_identity": original["numbers"] is restored["numbers"],
}
checks


In [ ]:
assert original == restored
assert original is not restored
assert original["numbers"] is not restored["numbers"]


# Problem 2 — Preserve shared references inside one object graph

Two dictionary keys will point to the **same list**.

After one `pickle.dumps(...)` / `pickle.loads(...)` round trip, prove that the alias relationship is preserved.

Then mutate through one alias and show the change through the other.


## Solution 2


In [ ]:
shared = ["alpha", "beta"]
graph = {
    "left": shared,
    "right": shared,
}

assert graph["left"] is graph["right"]

restored_graph = pickle.loads(
    pickle.dumps(graph, protocol=pickle.HIGHEST_PROTOCOL)
)

print("Alias preserved:", restored_graph["left"] is restored_graph["right"])

restored_graph["left"].append("gamma")
print(restored_graph["right"])

assert restored_graph["left"] is restored_graph["right"]
assert restored_graph["right"] == ["alpha", "beta", "gamma"]


The pickle memo records objects already encountered while serializing one graph. A later reference can point back to the previously serialized object instead of serializing another independent copy.


# Problem 3 — Separate pickles break cross-pickle aliasing

Suppose `outer["payload"]` references the same dictionary as `payload`.

Serialize `payload` and `outer` **separately**, restore both separately, and prove that the relationship `outer["payload"] is payload` no longer exists.

Then fix the design.


## Solution 3


In [ ]:
payload = {"count": 10}
outer = {"name": "container", "payload": payload}

payload_blob = pickle.dumps(payload, protocol=pickle.HIGHEST_PROTOCOL)
outer_blob = pickle.dumps(outer, protocol=pickle.HIGHEST_PROTOCOL)

payload2 = pickle.loads(payload_blob)
outer2 = pickle.loads(outer_blob)

print("Equal:", outer2["payload"] == payload2)
print("Identical:", outer2["payload"] is payload2)

assert outer2["payload"] == payload2
assert outer2["payload"] is not payload2


In [ ]:
# Fix: serialize the connected graph as ONE top-level object.
bundle = {
    "payload": payload,
    "outer": outer,
}

bundle2 = pickle.loads(
    pickle.dumps(bundle, protocol=pickle.HIGHEST_PROTOCOL)
)

assert bundle2["outer"]["payload"] is bundle2["payload"]
print("Fixed aliasing:", bundle2["outer"]["payload"] is bundle2["payload"])


# Problem 4 — Cyclic object graphs

Create a list that contains itself.

1. Pickle and unpickle it.
2. Verify that the restored list still points to itself.
3. Explain why a naïve recursive serializer could fail on this structure.


## Solution 4


In [ ]:
cycle = []
cycle.append("root")
cycle.append(cycle)

cycle2 = pickle.loads(
    pickle.dumps(cycle, protocol=pickle.HIGHEST_PROTOCOL)
)

print(cycle2[0])
print("Self-reference preserved:", cycle2[1] is cycle2)

assert cycle2[1] is cycle2


A naïve serializer that recursively expands every child could recurse forever. Pickle handles cycles by memoizing objects and emitting references back to objects already encountered.


# Problem 5 — Compare protocols, sizes, and opcodes

Serialize the same object with every available protocol from `0` through `pickle.HIGHEST_PROTOCOL`.

Collect:

- protocol number;
- byte length;
- first 12 bytes.

Then identify the smallest result on your Python version.


## Solution 5


In [ ]:
sample = {
    "text": "pickle " * 100,
    "values": list(range(500)),
    "flags": [True, False] * 100,
}

protocol_results = []

for protocol in range(pickle.HIGHEST_PROTOCOL + 1):
    data = pickle.dumps(sample, protocol=protocol)
    protocol_results.append(
        {
            "protocol": protocol,
            "bytes": len(data),
            "prefix": data[:12],
        }
    )

pprint(protocol_results)

smallest = min(protocol_results, key=lambda row: row["bytes"])
print("\nSmallest in this run:", smallest)


In [ ]:
# Inspect the newest protocol's pickle bytecode.
latest_blob = pickle.dumps(sample, protocol=pickle.HIGHEST_PROTOCOL)
pickletools.dis(latest_blob)


**Best practice:** normally use `protocol=pickle.HIGHEST_PROTOCOL` when both producer and consumer run compatible Python versions. If backward compatibility matters, choose the newest protocol supported by the oldest required Python runtime.


# Problem 6 — Custom state: exclude a transient cache

Implement a class whose expensive cache should **not** be serialized.

Requirements:

- persisted state: `name`, `values`;
- transient state: `_cache`;
- after unpickling, `_cache` must be empty;
- equality should compare the meaningful persisted state only.


## Solution 6


In [ ]:
@dataclass
class DataSeries:
    name: str
    values: list[float]
    _cache: dict[str, float] = field(default_factory=dict, repr=False, compare=False)

    def mean(self) -> float:
        if "mean" not in self._cache:
            self._cache["mean"] = sum(self.values) / len(self.values)
        return self._cache["mean"]

    def __getstate__(self):
        state = self.__dict__.copy()
        state.pop("_cache", None)
        return state

    def __setstate__(self, state):
        self.__dict__.update(state)
        self._cache = {}


series = DataSeries("latency_ms", [10.0, 20.0, 30.0])
assert series.mean() == 20.0
assert series._cache == {"mean": 20.0}

series2 = pickle.loads(
    pickle.dumps(series, protocol=pickle.HIGHEST_PROTOCOL)
)

print(series2)
print("Restored cache:", series2._cache)

assert series2 == series
assert series2._cache == {}
assert series2.mean() == 20.0


# Problem 7 — Versioned state migration

A class evolves over time.

Version 1 stored:

```python
{"version": 1, "first": "...", "last": "..."}
```

Version 2 stores:

```python
{"version": 2, "full_name": "..."}
```

Implement `__setstate__` so the current class can restore either shape.


## Solution 7


In [ ]:
class Customer:
    CURRENT_VERSION = 2

    def __init__(self, full_name: str):
        self.full_name = full_name

    def __getstate__(self):
        return {
            "version": self.CURRENT_VERSION,
            "full_name": self.full_name,
        }

    def __setstate__(self, state):
        version = state.get("version", 1)

        if version == 1:
            first = state["first"]
            last = state["last"]
            self.full_name = f"{first} {last}".strip()
        elif version == 2:
            self.full_name = state["full_name"]
        else:
            raise ValueError(f"Unsupported Customer state version: {version}")

    def __repr__(self):
        return f"Customer(full_name={self.full_name!r})"


In [ ]:
# Simulate unpickling legacy state by directly exercising migration logic.
legacy_state = {
    "version": 1,
    "first": "Ada",
    "last": "Lovelace",
}

customer = Customer.__new__(Customer)
customer.__setstate__(legacy_state)

print(customer)
assert customer.full_name == "Ada Lovelace"

# Confirm current objects still round-trip normally.
current = Customer("Grace Hopper")
current2 = pickle.loads(
    pickle.dumps(current, protocol=pickle.HIGHEST_PROTOCOL)
)

assert current2.full_name == "Grace Hopper"


**Best practice:** serialized application state should have an explicit schema/state version if you expect classes or structures to evolve.


# Problem 8 — What cannot be pickled directly?

Test these objects with `pickle.dumps`:

- a top-level function;
- a lambda;
- an open file handle.

Record which succeed on your Python runtime and explain why.


## Solution 8


In [ ]:
def square(x):
    return x * x


objects_to_test = {
    "top_level_function": square,
    "lambda": lambda x: x + 1,
}

for name, obj in objects_to_test.items():
    try:
        data = pickle.dumps(obj)
    except Exception as exc:
        print(f"{name:20} -> FAIL: {type(exc).__name__}: {exc}")
    else:
        print(f"{name:20} -> OK: {len(data)} bytes")


In [ ]:
with tempfile.TemporaryFile(mode="w+b") as fh:
    try:
        pickle.dumps(fh)
    except Exception as exc:
        print("open_file_handle      -> FAIL:", type(exc).__name__, exc)


Top-level functions are generally serialized **by module/name reference**, not by embedding their source code. Lambdas and many runtime resources such as open file handles are not normally picklable.

A pickled reference to a function or class also means the corresponding importable definition must exist when loading.


# Problem 9 — Restricted unpickling

Build an `Unpickler` subclass that only allows a tiny whitelist of globals.

Goal:

- ordinary primitive containers should load;
- `set` and `frozenset` are allowed;
- arbitrary user-defined classes are rejected.

This is a **defense-in-depth** technique, not a replacement for avoiding untrusted pickle.


## Solution 9


In [ ]:
class RestrictedUnpickler(pickle.Unpickler):
    SAFE_GLOBALS = {
        ("builtins", "set"): set,
        ("builtins", "frozenset"): frozenset,
    }

    def find_class(self, module, name):
        key = (module, name)
        if key in self.SAFE_GLOBALS:
            return self.SAFE_GLOBALS[key]
        raise pickle.UnpicklingError(
            f"global {module}.{name} is forbidden"
        )


def restricted_loads(data: bytes):
    return RestrictedUnpickler(io.BytesIO(data)).load()


In [ ]:
safe_obj = {
    "name": "demo",
    "numbers": [1, 2, 3],
    "tags": {"python", "pickle"},
}

safe_blob = pickle.dumps(
    safe_obj,
    protocol=pickle.HIGHEST_PROTOCOL
)

safe_restored = restricted_loads(safe_blob)
print(safe_restored)
assert safe_restored == safe_obj


In [ ]:
class CustomThing:
    def __init__(self, value):
        self.value = value


custom_blob = pickle.dumps(
    CustomThing(42),
    protocol=pickle.HIGHEST_PROTOCOL
)

try:
    restricted_loads(custom_blob)
except pickle.UnpicklingError as exc:
    print("Rejected as expected:", exc)
else:
    raise AssertionError("Custom class should have been rejected")


A whitelist is only as safe as the objects it permits. Some seemingly harmless callables can still enable unsafe behavior in the wrong reconstruction path, so keep the list extremely small.


# Problem 10 — Authenticate pickle bytes with HMAC before loading

You control both producer and consumer and have a secret key.

Design two functions:

- `seal_pickle(obj, key) -> bytes`
- `open_pickle(package, key) -> object`

Package format:

```text
32-byte SHA-256 HMAC || pickle bytes
```

Requirements:

1. verify with `hmac.compare_digest`;
2. reject tampered bytes before calling `pickle.loads`;
3. use the highest available protocol.


## Solution 10


In [ ]:
MAC_SIZE = hashlib.sha256().digest_size


def seal_pickle(obj: Any, key: bytes) -> bytes:
    payload = pickle.dumps(
        obj,
        protocol=pickle.HIGHEST_PROTOCOL,
    )
    tag = hmac.new(
        key,
        payload,
        hashlib.sha256,
    ).digest()
    return tag + payload


def open_pickle(package: bytes, key: bytes) -> Any:
    if len(package) < MAC_SIZE:
        raise ValueError("Package is too short")

    received_tag = package[:MAC_SIZE]
    payload = package[MAC_SIZE:]

    expected_tag = hmac.new(
        key,
        payload,
        hashlib.sha256,
    ).digest()

    if not hmac.compare_digest(received_tag, expected_tag):
        raise ValueError("Authentication failed")

    # Only deserialize AFTER successful authentication.
    return pickle.loads(payload)


In [ ]:
secret_key = b"example-training-key-change-me"
record = {"user_id": 123, "roles": ["reader", "writer"]}

package = seal_pickle(record, secret_key)
assert open_pickle(package, secret_key) == record

tampered = bytearray(package)
tampered[-1] ^= 0x01

try:
    open_pickle(bytes(tampered), secret_key)
except ValueError as exc:
    print("Tampering detected:", exc)
else:
    raise AssertionError("Tampering should have been detected")


**Important:** authentication only helps if the secret key is protected and the producer is trusted. If an attacker can produce a valid MAC, they can produce a malicious authenticated pickle too.


# Problem 11 — Atomic pickle-file writes

A program crashes while saving a large pickle. Writing directly to the destination can leave a partially written file.

Implement `atomic_pickle_dump(obj, path)` that:

1. creates a temporary file in the destination directory;
2. writes and flushes the pickle;
3. calls `os.fsync`;
4. replaces the destination atomically with `os.replace`;
5. cleans up the temporary file if an exception occurs.


## Solution 11


In [ ]:
def atomic_pickle_dump(obj: Any, path: str | os.PathLike) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    fd, temp_name = tempfile.mkstemp(
        prefix=f".{path.name}.",
        suffix=".tmp",
        dir=path.parent,
    )

    try:
        with os.fdopen(fd, "wb") as f:
            pickle.dump(
                obj,
                f,
                protocol=pickle.HIGHEST_PROTOCOL,
            )
            f.flush()
            os.fsync(f.fileno())

        os.replace(temp_name, path)

    except Exception:
        try:
            os.unlink(temp_name)
        except FileNotFoundError:
            pass
        raise


In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    target = Path(tmpdir) / "checkpoint.pkl"

    value = {
        "epoch": 7,
        "losses": [1.0, 0.8, 0.6],
    }

    atomic_pickle_dump(value, target)

    with target.open("rb") as f:
        loaded = pickle.load(f)

    print(loaded)
    assert loaded == value


# Problem 12 — Stream multiple pickle records

A single file can contain multiple pickle objects back-to-back.

Write:

- `append_record(file_obj, record)`
- `read_records(file_obj)`

`read_records` should yield objects until normal end-of-file.


## Solution 12


In [ ]:
def append_record(file_obj, record: Any) -> None:
    pickle.dump(
        record,
        file_obj,
        protocol=pickle.HIGHEST_PROTOCOL,
    )


def read_records(file_obj):
    while True:
        try:
            yield pickle.load(file_obj)
        except EOFError:
            return


In [ ]:
buffer = io.BytesIO()

for i in range(5):
    append_record(
        buffer,
        {"sequence": i, "square": i * i},
    )

buffer.seek(0)
records = list(read_records(buffer))

pprint(records)

assert [r["sequence"] for r in records] == list(range(5))


For durable production logs, a length-prefixed or framed format is often easier to recover after corruption. Back-to-back pickle objects are convenient but do not provide built-in checksums, indexing, or robust corruption recovery.


# Problem 13 — Persistent IDs for external objects

Imagine large binary blobs live in an external object store and should not be duplicated inside every pickle.

Implement custom `Pickler` / `Unpickler` classes:

- `BlobRef(key)` objects are replaced by persistent IDs;
- loading resolves those IDs from a provided dictionary.


## Solution 13


In [ ]:
@dataclass(frozen=True)
class BlobRef:
    key: str


class ExternalizingPickler(pickle.Pickler):
    def persistent_id(self, obj):
        if isinstance(obj, BlobRef):
            return ("BLOB", obj.key)
        return None


class ResolvingUnpickler(pickle.Unpickler):
    def __init__(self, file, blob_store):
        super().__init__(file)
        self.blob_store = blob_store

    def persistent_load(self, pid):
        kind, key = pid

        if kind != "BLOB":
            raise pickle.UnpicklingError(
                f"Unsupported persistent object: {pid!r}"
            )

        try:
            return self.blob_store[key]
        except KeyError as exc:
            raise pickle.UnpicklingError(
                f"Missing external blob: {key!r}"
            ) from exc


In [ ]:
blob_store = {
    "img-001": b"\x89PNG...fake-training-bytes...",
    "doc-007": b"%PDF-fake-training-bytes...",
}

document = {
    "title": "External references",
    "thumbnail": BlobRef("img-001"),
    "attachment": BlobRef("doc-007"),
}

buffer = io.BytesIO()

ExternalizingPickler(
    buffer,
    protocol=pickle.HIGHEST_PROTOCOL,
).dump(document)

print("Pickle bytes:", len(buffer.getvalue()))

buffer.seek(0)
restored_document = ResolvingUnpickler(
    buffer,
    blob_store,
).load()

pprint(restored_document)

assert restored_document["thumbnail"] == blob_store["img-001"]
assert restored_document["attachment"] == blob_store["doc-007"]


# Problem 14 — Pickle plus compression

Serialize a repetitive object, then compare:

- raw pickle;
- gzip-compressed pickle;
- bzip2-compressed pickle.

Check that decompression plus unpickling restores the original object.


## Solution 14


In [ ]:
repetitive = {
    "events": [
        {
            "type": "page_view",
            "path": "/products/advanced-pickle",
            "country": "BG",
        }
        for _ in range(10_000)
    ]
}

raw = pickle.dumps(
    repetitive,
    protocol=pickle.HIGHEST_PROTOCOL,
)

gz = gzip.compress(raw)
bz = bz2.compress(raw)

sizes = {
    "raw_pickle": len(raw),
    "gzip": len(gz),
    "bzip2": len(bz),
}

pprint(sizes)

assert pickle.loads(gzip.decompress(gz)) == repetitive
assert pickle.loads(bz2.decompress(bz)) == repetitive


Compression is useful for repetitive payloads, but it costs CPU. Measure with representative data instead of assuming the smallest file is always the best engineering choice.


# Problem 15 — Benchmark protocol and compression trade-offs

Benchmark serialization and deserialization using `timeit`.

Compare:

1. pickle only;
2. pickle + gzip;
3. pickle + bzip2.

Use enough repetitions to get a stable rough comparison on your machine.


## Solution 15


In [ ]:
benchmark_obj = {
    "rows": [
        (i, f"user-{i % 100}", i % 7, i * 0.25)
        for i in range(20_000)
    ]
}


def pickle_only():
    data = pickle.dumps(
        benchmark_obj,
        protocol=pickle.HIGHEST_PROTOCOL,
    )
    return pickle.loads(data)


def pickle_gzip():
    data = pickle.dumps(
        benchmark_obj,
        protocol=pickle.HIGHEST_PROTOCOL,
    )
    packed = gzip.compress(data)
    return pickle.loads(gzip.decompress(packed))


def pickle_bzip2():
    data = pickle.dumps(
        benchmark_obj,
        protocol=pickle.HIGHEST_PROTOCOL,
    )
    packed = bz2.compress(data)
    return pickle.loads(bz2.decompress(packed))


benchmarks = {}

for name, fn in [
    ("pickle_only", pickle_only),
    ("pickle_gzip", pickle_gzip),
    ("pickle_bzip2", pickle_bzip2),
]:
    runs = timeit.repeat(fn, repeat=5, number=3)
    benchmarks[name] = {
        "best_seconds": min(runs),
        "median_seconds": statistics.median(runs),
    }

pprint(benchmarks)


# Problem 16 — Optimize a pickle stream

`pickletools.optimize` can remove some unnecessary memo operations from an already generated pickle.

Measure whether it changes the byte size for a sample object, then verify that the optimized pickle loads to an equal value.


## Solution 16


In [ ]:
obj = [
    {"x": i, "label": f"item-{i % 10}"}
    for i in range(2_000)
]

before = pickle.dumps(obj, protocol=pickle.HIGHEST_PROTOCOL)
after = pickletools.optimize(before)

print("Before:", len(before))
print("After: ", len(after))
print("Saved: ", len(before) - len(after))

assert pickle.loads(after) == obj


Do not assume optimization will always save much space. Modern protocols and common object graphs may already be compact. Measure before introducing extra processing.


# Problem 17 — Class evolution and import-path fragility

Pickle generally stores references to classes and functions by module and qualified name.

Answer these questions:

1. Why can renaming or moving a class break old pickles?
2. Why can notebooks make long-lived pickles especially fragile?
3. What design choices reduce that risk?


## Solution 17

1. A pickle may record something conceptually like `package.module.Customer`. During loading, Python imports/locates that global. If the class moved or was renamed, the reference may no longer resolve.
2. Notebook-defined classes often live under `__main__`, and the exact interactive environment that created them may not exist later.
3. Better long-lived designs include:
   - put serialized classes in stable importable modules;
   - keep compatibility shims when moving classes;
   - store explicit state versions;
   - serialize plain data structures for long-term persistence;
   - choose a language-neutral format when interoperability or long retention matters.

Pickle is excellent for trusted Python-to-Python persistence, but it is not automatically a durable archival schema.


# Problem 18 — Dataclass with invariants after unpickling

Create an `Account` dataclass with these rules:

- `balance` must be non-negative;
- `transactions` is a list of integer deltas;
- `_transaction_count` is derived and should not be persisted;
- unpickling must revalidate the invariant.

Then demonstrate successful round-trip and rejection of invalid state supplied to `__setstate__`.


## Solution 18


In [ ]:
@dataclass
class Account:
    owner: str
    balance: int
    transactions: list[int] = field(default_factory=list)
    _transaction_count: int = field(init=False, repr=False)

    def __post_init__(self):
        self._validate()
        self._transaction_count = len(self.transactions)

    def _validate(self):
        if self.balance < 0:
            raise ValueError("balance must be non-negative")

    def __getstate__(self):
        return {
            "owner": self.owner,
            "balance": self.balance,
            "transactions": self.transactions,
        }

    def __setstate__(self, state):
        self.owner = state["owner"]
        self.balance = state["balance"]
        self.transactions = list(state["transactions"])
        self._validate()
        self._transaction_count = len(self.transactions)


In [ ]:
account = Account(
    owner="Ada",
    balance=120,
    transactions=[100, 20],
)

account2 = pickle.loads(
    pickle.dumps(account, protocol=pickle.HIGHEST_PROTOCOL)
)

print(account2)
print("Transaction count:", account2._transaction_count)

assert account2.balance == 120
assert account2._transaction_count == 2


In [ ]:
bad_state = {
    "owner": "Mallory",
    "balance": -1,
    "transactions": [],
}

broken = Account.__new__(Account)

try:
    broken.__setstate__(bad_state)
except ValueError as exc:
    print("Invalid state rejected:", exc)
else:
    raise AssertionError("Invalid state should be rejected")


# Problem 19 — Capstone: robust local checkpoint manager

Build a small checkpoint format for **trusted local application state**.

Requirements:

- store a format version;
- store arbitrary application state;
- authenticate the serialized payload with HMAC-SHA256;
- write atomically;
- verify authentication before deserializing;
- reject unsupported checkpoint versions.

The file format will be:

```text
32-byte HMAC || pickle({"format_version": 1, "state": ...})
```


## Solution 19


In [ ]:
CHECKPOINT_VERSION = 1
CHECKPOINT_MAC_SIZE = hashlib.sha256().digest_size


def build_checkpoint_bytes(state: Any, key: bytes) -> bytes:
    envelope = {
        "format_version": CHECKPOINT_VERSION,
        "state": state,
    }

    payload = pickle.dumps(
        envelope,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

    tag = hmac.new(
        key,
        payload,
        hashlib.sha256,
    ).digest()

    return tag + payload


def parse_checkpoint_bytes(package: bytes, key: bytes) -> Any:
    if len(package) < CHECKPOINT_MAC_SIZE:
        raise ValueError("Checkpoint is truncated")

    tag = package[:CHECKPOINT_MAC_SIZE]
    payload = package[CHECKPOINT_MAC_SIZE:]

    expected = hmac.new(
        key,
        payload,
        hashlib.sha256,
    ).digest()

    if not hmac.compare_digest(tag, expected):
        raise ValueError("Checkpoint authentication failed")

    envelope = pickle.loads(payload)

    if envelope.get("format_version") != CHECKPOINT_VERSION:
        raise ValueError(
            f"Unsupported checkpoint version: "
            f"{envelope.get('format_version')!r}"
        )

    return envelope["state"]


def save_checkpoint(
    state: Any,
    path: str | os.PathLike,
    key: bytes,
) -> None:
    package = build_checkpoint_bytes(state, key)

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    fd, temp_name = tempfile.mkstemp(
        prefix=f".{path.name}.",
        suffix=".tmp",
        dir=path.parent,
    )

    try:
        with os.fdopen(fd, "wb") as f:
            f.write(package)
            f.flush()
            os.fsync(f.fileno())

        os.replace(temp_name, path)

    except Exception:
        try:
            os.unlink(temp_name)
        except FileNotFoundError:
            pass
        raise


def load_checkpoint(
    path: str | os.PathLike,
    key: bytes,
) -> Any:
    package = Path(path).read_bytes()
    return parse_checkpoint_bytes(package, key)


In [ ]:
checkpoint_key = b"training-only-secret"

app_state = {
    "step": 250,
    "metrics": {
        "loss": 0.123,
        "accuracy": 0.987,
    },
    "pending_jobs": ["job-17", "job-21"],
}

with tempfile.TemporaryDirectory() as tmpdir:
    checkpoint_path = Path(tmpdir) / "state.chk"

    save_checkpoint(
        app_state,
        checkpoint_path,
        checkpoint_key,
    )

    recovered = load_checkpoint(
        checkpoint_path,
        checkpoint_key,
    )

    pprint(recovered)
    assert recovered == app_state


In [ ]:
# Tampering test for the capstone format.
package = build_checkpoint_bytes(
    app_state,
    checkpoint_key,
)

tampered = bytearray(package)
tampered[-2] ^= 0xFF

try:
    parse_checkpoint_bytes(
        bytes(tampered),
        checkpoint_key,
    )
except ValueError as exc:
    print("Rejected:", exc)
else:
    raise AssertionError("Tampered checkpoint must be rejected")


# Bonus challenge A — Predict before running

For each expression, predict `True` or `False`, then execute the cell.

The key idea: **identity relationships inside one serialized graph can be preserved, but identities to pre-existing external objects cannot be preserved across a round trip.**


In [ ]:
shared = {"n": 1}
root = [shared, shared]

restored = pickle.loads(
    pickle.dumps(root, protocol=pickle.HIGHEST_PROTOCOL)
)

predictions = {
    "same_top_level": root is restored,
    "same_original_child": root[0] is restored[0],
    "restored_alias_preserved": restored[0] is restored[1],
    "values_equal": root == restored,
}

pprint(predictions)


# Bonus challenge B — One pickle stream, two dumps, shared memo?

Compare these two approaches:

### Approach 1
Call `pickle.dump(a, file)` and then `pickle.dump(b, file)`.

### Approach 2
Create **one `pickle.Pickler(file)` instance** and call `.dump(a)` then `.dump(b)`.

Investigate whether the same memo can preserve a shared reference across the two dumped top-level objects.


## Solution — Bonus B


In [ ]:
shared = {"shared": True}
a = {"left": shared}
b = {"right": shared}

# Approach 1: convenience function creates a fresh Pickler each call.
buf1 = io.BytesIO()
pickle.dump(a, buf1, protocol=pickle.HIGHEST_PROTOCOL)
pickle.dump(b, buf1, protocol=pickle.HIGHEST_PROTOCOL)

buf1.seek(0)
a1 = pickle.load(buf1)
b1 = pickle.load(buf1)

print("Approach 1 shared:", a1["left"] is b1["right"])


In [ ]:
# Approach 2: reuse ONE Pickler, so its memo survives across dump calls.
buf2 = io.BytesIO()
p = pickle.Pickler(buf2, protocol=pickle.HIGHEST_PROTOCOL)
p.dump(a)
p.dump(b)

buf2.seek(0)

# Reuse ONE Unpickler so its memo also survives across load calls.
u = pickle.Unpickler(buf2)
a2 = u.load()
b2 = u.load()

print("Approach 2 shared:", a2["left"] is b2["right"])

assert a2["left"] is b2["right"]


This is a subtle but powerful distinction: memoization belongs to a `Pickler` / `Unpickler` instance, not to the file itself.


# Best-practices checklist

Use this checklist when deciding whether and how to use pickle:

- **Trust boundary:** never load unauthenticated data from an untrusted source.
- **Authentication:** if trusted systems exchange pickles, authenticate bytes *before* loading.
- **Stable definitions:** keep serialized classes/functions in stable importable modules.
- **Version state:** include a state/schema version for data that may outlive current code.
- **One graph:** serialize connected objects together when aliasing/identity relationships matter.
- **Transient fields:** exclude caches, locks, open handles, sockets, and recomputable data.
- **Validate on restore:** `__setstate__` should re-establish invariants.
- **Atomic files:** write to a temporary file and replace the destination.
- **Compatibility:** select a protocol supported by every required Python runtime.
- **Compression:** benchmark it; do not assume smaller always means better.
- **Inspection:** use `pickletools` to study pickle bytecode without deserializing it.
- **Interchange:** prefer a safer language-neutral format for external APIs and untrusted data.


# Final review questions

Try answering these without running code first:

1. Why can `a == b` be `True` while `a is b` is `False` after unpickling?
2. When does pickle preserve aliases?
3. Why can two separately created pickle blobs lose a relationship between objects?
4. What problem does the pickle memo solve?
5. Why is `pickle.loads(untrusted_bytes)` unsafe?
6. Why must an HMAC be verified *before* unpickling?
7. What does `__getstate__` let you exclude?
8. What does `__setstate__` let you repair or migrate?
9. Why can moving a class to another module break old pickle files?
10. Why are atomic writes useful for checkpoints?
11. What are persistent IDs useful for?
12. Why should you benchmark compression instead of blindly enabling it?


## Concise answers

1. Equality compares value/state; identity asks whether two references point to the exact same object.
2. Aliases are preserved among objects serialized within the same pickle memo/object graph.
3. Independent serialization runs have independent memos and reconstruct independent graphs.
4. The memo supports repeated references and cycles without duplicating or infinitely recursing.
5. Unpickling can invoke globals/callables during object reconstruction.
6. Verification must happen first so attacker-modified bytes never reach the dangerous deserializer.
7. Transient or recomputable attributes such as caches.
8. Invariants, defaults, derived fields, and migrations from older state layouts.
9. Pickles often identify classes/functions by importable module and qualified name.
10. They reduce the chance that a crash leaves a partially written destination file.
11. Replacing selected objects with references resolved from an external store.
12. Compression trades file size for CPU time and sometimes latency/memory.
